# Data Cleaning & Transformation

## Objective

Prepare the audited raw dataset for downstream SQL analysis, statistical analysis,
and Power BI while preserving the original raw data.

## Cleaning principles

1. Raw files will never be overwritten.
2. Only confirmed data-quality issues will be corrected.
3. Missing values will be handled according to business meaning.
4. Potential business-rule issues will be flagged rather than assumed.
5. Every transformation will be documented.
6. The cleaned dataset will be re-audited before analysis.

# Import Libraries

In [55]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

In [56]:
RAW_PATH = Path(r"D:\Profit_leak_growth_Analysis\data\raw")
CLEAN_PATH = Path(r"D:\Profit_leak_growth_Analysis\data\cleaned")

CLEAN_PATH.mkdir(parents=True, exist_ok=True)

customers = pd.read_csv(RAW_PATH / "raw_customers.csv")
products = pd.read_csv(RAW_PATH / "raw_products.csv")
stores = pd.read_csv(RAW_PATH / "raw_stores.csv")
campaigns = pd.read_csv(RAW_PATH / "raw_campaigns.csv")
orders = pd.read_csv(RAW_PATH / "raw_orders.csv")
order_items = pd.read_csv(RAW_PATH / "raw_order_items.csv")
returns = pd.read_csv(RAW_PATH / "raw_returns.csv")
shipments = pd.read_csv(RAW_PATH / "raw_shipments.csv")
marketing = pd.read_csv(RAW_PATH / "raw_marketing_daily.csv")
date_dim = pd.read_csv(RAW_PATH / "dim_date.csv")

In [57]:
customers.head()

,customer_id,signup_date,acquisition_channel,customer_type,country,state,city,age_band,income_band,loyalty_tier
0,C000001,2023-09-13,Organic,Small Business,USA,Georgia,Atlanta,25-34,150K+,Gold
1,C000002,2023-05-13,Direct,Small Business,USA,Illinois,Chicago,55-64,100K-150K,Silver
2,C000003,2024-05-12,Meta,Consumer,USA,Washington,Seattle,45-54,60K-75K,Bronze
3,C000004,2024-01-30,Email,Consumer,USA,Texas,Dallas,65+,40K-60K,Silver
4,C000005,2024-06-28,Email,Consumer,USA,New York,New York,35-44,75K-100K,Bronze


# Creating a cleaning_log

The purpose of this is to ans "What did I change, why did I change it, and how many records were affected?"

In [58]:
#Create a cleaning log
cleaning_log = []

def log_cleaning_issue(
    table,
    issue,
    action,
    rows_affected,
    rationale,
    status="Applied"
):
    cleaning_log.append({
        "table": table,
        "issue": issue,
        "action": action,
        "rows_affected": rows_affected,
        "rationale": rationale,
        "status": status
    })

#Creating a csv of it
cleaning_log_df = pd.DataFrame(cleaning_log)

# Add a file name and .csv extension to the path
cleaning_log_df.to_csv(
    r"D:\Profit_leak_growth_Analysis\documentation\cleaning_log.csv",
    index=False
)

# STEP 1: Fix date column (Datetype from obj to datetime)

In [59]:
#Customers table
customers['signup_date']=pd.to_datetime(customers['signup_date'], errors='coerce' )

#products
products["launch_date"] = pd.to_datetime(
    products["launch_date"],
    errors="coerce"
)

#Stores
stores["opened_date"] = pd.to_datetime(
    stores["opened_date"],
    errors="coerce"
)

#Campaigns
campaigns['start_date']=pd.to_datetime( campaigns['start_date'], errors='coerce' )

#Orders
orders["order_date"] = pd.to_datetime(
    orders["order_date"],
    errors="coerce"
)

#Returns
returns["return_date"] = pd.to_datetime(
    returns["return_date"],
    errors="coerce"
)

#Shipments
shipments["ship_date"] = pd.to_datetime(
    shipments["ship_date"],
    errors="coerce"
)

shipments["promised_date"] = pd.to_datetime(
    shipments["promised_date"],
    errors="coerce"
)

shipments["delivery_date"] = pd.to_datetime(
    shipments["delivery_date"],
    errors="coerce"
)

#Marketing
marketing["date"] = pd.to_datetime(
    marketing["date"],
    errors="coerce"
)

#Date Dimension (dim)
date_dim["date"] = pd.to_datetime(
    date_dim["date"],
    errors="coerce"
)

In [60]:
#Validate the dates that is it converted to date datatype or not
date_columns_check = {
    "customers": ["signup_date"],
    "products": ["launch_date"],
    "stores": ["opened_date"],
    "campaigns": ["start_date"],
    "orders": ["order_date"],
    "returns": ["return_date"],
    "shipments": ["ship_date", "promised_date","delivery_date"],
    "marketing": ["date"],
    "date_dim": ["date"]
}

for table_name, columns in date_columns_check.items():
    df = {
        "customers": customers,
        "products": products,
        "stores": stores,
        "campaigns": campaigns,
        "orders": orders,
        "returns": returns,
        "shipments": shipments,
        "marketing": marketing,
        "date_dim": date_dim
    }[table_name]

    for column in columns:
        print(table_name, column, df[column].dtype)

customers signup_date datetime64[ns]
products launch_date datetime64[ns]
stores opened_date datetime64[ns]
campaigns start_date datetime64[ns]
orders order_date datetime64[ns]
returns return_date datetime64[ns]
shipments ship_date datetime64[ns]
shipments promised_date datetime64[ns]
shipments delivery_date datetime64[ns]
marketing date datetime64[ns]
date_dim date datetime64[ns]


#### As we can see clearly the datetypes are succesfully converted into datetime.

In [61]:
#Now we will check is their missing value in date column
for table_name, columns in date_columns_check.items():
    df = {
        "customers": customers,
        "products": products,
        "stores": stores,
        "campaigns": campaigns,
        "orders": orders,
        "returns": returns,
        "shipments": shipments,
        "marketing": marketing,
        "date_dim": date_dim
    }[table_name]

    for column in columns:
        nat_count = df[column].isna().sum()
        print(table_name, column, nat_count)

customers signup_date 0
products launch_date 0
stores opened_date 0
campaigns start_date 0
orders order_date 0
returns return_date 0
shipments ship_date 0
shipments promised_date 0
shipments delivery_date 0
marketing date 0
date_dim date 0


# Step 2: Acquisition channel standardization

Our canonical values are:
* Organic
* Direct
* Meta
* Email
* Affiliate
* Google Ads

In [62]:
#First we will see the values
customers["acquisition_channel"].value_counts()

acquisition_channel
Google Ads    6629
Meta          5949
Organic       5330
Direct        4801
Email         4174
Affiliate     3004
Google_Ads      12
GOOGLE ADS      11
ORGANIC         10
DIRECT          10
meta            10
google ads      10
email            9
direct           9
META             9
EMAIL            8
organic          8
affiliate        4
AFFILIATE        3
Name: count, dtype: int64

In [63]:
#Create mapping
acquisition_mapping = {
    "META": "Meta",
    "meta": "Meta",

    "DIRECT": "Direct",
    "direct": "Direct",

    "ORGANIC": "Organic",
    "organic": "Organic",

    "EMAIL": "Email",
    "email": "Email",

    "AFFILIATE": "Affiliate",
    "affiliate": "Affiliate",

    "GOOGLE ADS": "Google Ads",
    "google ads": "Google Ads",
    "Google_Ads": "Google Ads"
}

#Now replace values
customers["acquisition_channel"] = (
    customers["acquisition_channel"]
    .replace(acquisition_mapping)
)

#Verify
customers["acquisition_channel"].value_counts()

acquisition_channel
Google Ads    6662
Meta          5968
Organic       5348
Direct        4820
Email         4191
Affiliate     3011
Name: count, dtype: int64

In [64]:
# Now log it into cleaning log
log_cleaning_issue(
    table="raw_customers",
    issue="Acquisition channel formatting variants",
    action="Standardized casing and Google Ads formatting to canonical labels",
    rows_affected=113,
    rationale="Values represented the same acquisition channels with inconsistent casing/formatting."
)

# STEP 3: HANDLING MISSING VALUES

### Store_id missing value

We have `56358 missing store_ids` in orders table. But It is not a major issue because we studied early that online orders are not neccesaryly supposed to have a store_id , so online orders store_id missing is not a major issue.

* Physical Store → store_id required
* Website/Mobile App/Marketplace → store_id should be NULL

So we don't fill that values.

"Missingness was evaluated according to business meaning rather than treated as a universal data-quality defect."

### Coupon code missing values

We have `37,660 missing coupon codes` in orders table. A NULL coupon code can legitimately mean: `No coupon used`. So we don't fill these missing values with any value.

### Missing customer city

We  have `240 missing city value`s in customers table. So we don't delete these null values and we also don't fill them with any city on our own .Because we don't know their city.

Later, when geography is required, we can either:
* exclude unknown cities from city-level analysis, or
* report them as "Unknown" in an analytical presentation layer.
But not invent information.

So we keep that null values same (means null).


### Missing income band

We have `90 missing values in income band` in customers table. We keep that values same. Means we don't fill or delete them. Income is particularly sensitive to arbitrary imputation because it could distort customer segmentation.

### Missing Payment method

We have `450 missing payment_method` in customers table.
Keep NULL.

We can later report:
* Payment method unknown = 450

if it matters to the analysis.

### Missing Marketing clicks

In [65]:
marketing[marketing["clicks"].isna()]

,date,campaign_id,impressions,clicks,ad_spend,new_customers,attributed_orders,attributed_revenue
548,2025-03-15,CAM010,376284,NaN,8603.05,162,356,19411.24
1000,2025-01-21,CAM032,226982,NaN,2773.20,368,804,72511.46
1362,2024-07-23,CAM023,60651,NaN,914.26,16,38,3353.86
6366,2025-08-29,CAM017,313092,NaN,4785.86,811,1663,109179.53
6790,2025-03-12,CAM021,253766,NaN,3927.57,410,757,83263.02
6899,2025-03-02,CAM029,89305,NaN,871.89,47,125,12738.15
7133,2024-04-05,CAM051,135105,NaN,3213.04,52,110,12689.15


We know these are missing values, not zero clicks.

So we don't fill these missing values as 0, because we don't have any evidence that the campaign have zero clicks. So we keep them same/missing.

In [66]:
# But we have to change its typpe, means clicks column type. Convert it into integer
marketing["clicks"] = marketing["clicks"].astype("Int64")

# STEP 4: MARKETING GRAIN

Data:
* 7,965 rows originally
* 5,644 unique date + campaign_id combinations
* multiple records for the same campaign/date
* 1,806 duplicate grain groups


Our intended grain is:
* 1 row = 1 campaign + 1 date

But we found
* 2,321 excess rows
* 1,806 duplicate date + campaign groups

And importantly:

* Exact duplicate rows = 0.

So this is NOT a normal drop_duplicates() problem.

In [67]:
# So instead of directly dropping them, first we: quarantine the marketing duplicates

marketing_duplicate_groups = (
    marketing
    .groupby(["date", "campaign_id"])
    .size()
    .reset_index(name="row_count")
)

marketing_duplicate_groups = marketing_duplicate_groups[
    marketing_duplicate_groups["row_count"] > 1
]

marketing_duplicate_groups.head(10)

,date,campaign_id,row_count
2,2024-01-13,CAM022,2
3,2024-01-14,CAM013,2
5,2024-01-15,CAM049,2
15,2024-01-20,CAM022,2
22,2024-01-25,CAM049,2
24,2024-01-26,CAM022,2
28,2024-01-29,CAM013,2
30,2024-01-29,CAM049,3
34,2024-01-30,CAM049,2
36,2024-01-31,CAM013,2


In [68]:
#It give us all that duplicate rows and that complete data
marketing_duplicate_rows = marketing[marketing.duplicated(["date", "campaign_id"], keep=False)].sort_values(["date", "campaign_id"])
marketing_duplicate_rows.head()

,date,campaign_id,impressions,clicks,ad_spend,new_customers,attributed_orders,attributed_revenue
2213,2024-01-13,CAM022,285896,6044,3462.80,108,205,20354.47
5697,2024-01-13,CAM022,235733,3489,3095.42,84,148,14220.03
3319,2024-01-14,CAM013,131690,6906,2284.62,221,422,28382.46
4605,2024-01-14,CAM013,297508,11055,3418.14,410,976,107765.93
2671,2024-01-15,CAM049,247634,1890,1715.46,11,29,2371.28


We are aggregating them, because duplicate rows have same columns and their is no specific additional column, so we aggregate the duplicate rows that are of same dates.

Aggregate them. Why? Because all of these fields are additive daily campaign metrics:
* impressions
* clicks
* ad_spend
* new_customers
* attributed_orders
* attributed_revenue

If two rows represent the same campaign on the same date and there is no additional dimension, then keeping two rows violates our declared table grain.



In [69]:
#Aggregating the metrices
marketing_clean = (
    marketing
    .groupby(["date", "campaign_id"], as_index=False)
    .agg({
        "impressions": "sum",
        "clicks": "sum",
        "ad_spend": "sum",
        "new_customers": "sum",
        "attributed_orders": "sum",
        "attributed_revenue": "sum"
    })
)

# Now check the shape (It should be 5644 unique combinations now)
marketing_clean.shape

(5644, 8)

In [70]:
#Now we check is it still any duplicate left or not
marketing_clean.duplicated(
    ["date", "campaign_id"]
).sum()

np.int64(0)

In [71]:
marketing_clean["clicks"].le(
    marketing_clean["impressions"]
).all()

np.True_

Marketing daily records were aggregated to the intended grain of one row per campaign per day. Additive campaign metrics were summed across duplicate campaign-date records. This resolves a synthetic generation/grain issue rather than deleting records.

# STEP 5: The 51 return violations

We have :
* 51 order items
* 102 return records
* 51 excess units

we don't know which order return is valid and which is an error, so.
 
As this is a stimulated data, so our intended rule is: 
* `One physical unit can only be refunded once.`

#### So what we are doing now
`Keep the earliest valid return event.`

For each affected order_item_id:

* Sort returns by return_date
* Allow returned quantity until purchased quantity is reached
* Any excess quantity becomes invalid for analytical return quantity
* Preserve the original record
* Flag the excess record

This is better than silently deleting data.

In [72]:
# Sort
returns_clean = returns.sort_values(
    ["order_item_id", "return_date", "return_id"]
).copy()

#Bring purchased quantity into the table:
returns_clean = returns_clean.merge(
    order_items[
        ["order_item_id", "quantity"]
    ],
    on="order_item_id",
    how="left"
)

#Then calculate cumulative returned quantity:
returns_clean["cumulative_return_quantity"] = (
    returns_clean
    .groupby("order_item_id")["return_quantity"]
    .cumsum()
)

#Return quantity validation
returns_clean["valid_return_quantity"] = np.where(
    returns_clean["cumulative_return_quantity"]
    <= returns_clean["quantity"],
    returns_clean["return_quantity"],
    np.maximum(
        0,
        returns_clean["quantity"]
        - (
            returns_clean["cumulative_return_quantity"]
            - returns_clean["return_quantity"]
        )
    )
)

#Flag
returns_clean["return_quantity_issue"] = (
    returns_clean["valid_return_quantity"]
    != returns_clean["return_quantity"]
)

### Refund amount

This is important.

If a return quantity becomes invalid/zero, we shouldn't continue treating its refund as a valid refund.

In [73]:
returns_clean["valid_refund_amount"] = np.where(
    returns_clean["valid_return_quantity"] > 0,
    returns_clean["refund_amount"],
    0
)

returns_clean["valid_return_shipping_cost"] = np.where(
    returns_clean["valid_return_quantity"] > 0,
    returns_clean["return_shipping_cost"],
    0
)

returns_clean["valid_restocking_cost"] = np.where(
    returns_clean["valid_return_quantity"] > 0,
    returns_clean["restocking_cost"],
    0
)

# STEP 6: The Welcome 10 issue

We discovered:

`WELCOME10`, appears with: "discount_rate = 5%" in 14,236 orders.

So we are not changing:

5% → 10%

just because the coupon name says "10".

Instead, we create a documentation item:

* `Issue`:
WELCOME10 appears with a 5% discount rate.

* `Classification`:
Business-rule review.

* `Action`:
No automatic correction.

* `Reason`:
Coupon naming alone does not prove that the discount rate is incorrect.



# Save all the cleaned files

In [74]:
# Save all the cleaned and transformed files
customers.to_csv(
    CLEAN_PATH / "customers_clean.csv",
    index=False
)

products.to_csv(
    CLEAN_PATH / "products_clean.csv",
    index=False
)

stores.to_csv(
    CLEAN_PATH / "stores_clean.csv",
    index=False
)

campaigns.to_csv(
    CLEAN_PATH / "campaigns_clean.csv",
    index=False
)

orders.to_csv(
    CLEAN_PATH / "orders_clean.csv",
    index=False
)

order_items.to_csv(
    CLEAN_PATH / "order_items_clean.csv",
    index=False
)

# FIXED: Saved 'returns_clean' to preserve validation flags and valid quantities
returns_clean.to_csv(
    CLEAN_PATH / "returns_clean.csv",
    index=False
)

shipments.to_csv(
    CLEAN_PATH / "shipments_clean.csv",
    index=False
)

# FIXED: Saved 'marketing_clean' to preserve the aggregated unique grain
marketing_clean.to_csv(
    CLEAN_PATH / "marketing_clean.csv",
    index=False
)

date_dim.to_csv(
    CLEAN_PATH / "dim_date_clean.csv",
    index=False
)

# Post Cleaning Validation

In [75]:
# Load the clean files
customers_clean = pd.read_csv(CLEAN_PATH / "customers_clean.csv")
products_clean = pd.read_csv(CLEAN_PATH / "products_clean.csv")
stores_clean = pd.read_csv(CLEAN_PATH / "stores_clean.csv")
campaigns_clean = pd.read_csv(CLEAN_PATH / "campaigns_clean.csv")
orders_clean = pd.read_csv(CLEAN_PATH / "orders_clean.csv")
order_items_clean = pd.read_csv(CLEAN_PATH / "order_items_clean.csv")
returns_clean = pd.read_csv(CLEAN_PATH / "returns_clean.csv")
shipments_clean = pd.read_csv(CLEAN_PATH / "shipments_clean.csv")
marketing_clean = pd.read_csv(CLEAN_PATH / "marketing_clean.csv")
date_dim_clean = pd.read_csv(CLEAN_PATH / "dim_date_clean.csv")


In [76]:
#Acquisition channels
customers_clean["acquisition_channel"].value_counts()

acquisition_channel
Google Ads    6662
Meta          5968
Organic       5348
Direct        4820
Email         4191
Affiliate     3011
Name: count, dtype: int64

In [77]:
#Marketing Grain: (It should be 0)
marketing_clean.duplicated(
    ["date", "campaign_id"]
).sum()

np.int64(0)

In [80]:
#Check rows and column
marketing_clean.shape

(5644, 8)

# Marketing Funnel

In [83]:
#It should return a empty dataframe
marketing_clean[
    marketing_clean["clicks"] >
    marketing_clean["impressions"]
]

,date,campaign_id,impressions,clicks,ad_spend,new_customers,attributed_orders,attributed_revenue


In [84]:
#It should return 0
marketing_clean[
    marketing_clean[
        [
            "impressions",
            "clicks",
            "ad_spend",
            "new_customers",
            "attributed_orders",
            "attributed_revenue"
        ]
    ] < 0
].sum()

date                    0
campaign_id             0
impressions           0.0
clicks                0.0
ad_spend              0.0
new_customers         0.0
attributed_orders     0.0
attributed_revenue    0.0
dtype: object

# Return Quantity

In [87]:
returned_totals = (
    returns_clean
    .groupby("order_item_id")["valid_return_quantity"]
    .sum()
)

In [90]:
#It should return 0
return_validation = (
    order_items[
        ["order_item_id", "quantity"]
    ]
    .merge(
        returned_totals,
        on="order_item_id",
        how="left"
    )
)

return_validation["valid_return_quantity"] = (
    return_validation["valid_return_quantity"]
    .fillna(0)
)

invalid_after_cleaning = return_validation[
    return_validation["valid_return_quantity"]
    > return_validation["quantity"]
]

len(invalid_after_cleaning)

0

# Financial logic

In [92]:
#It should return 0
(
    order_items["gross_revenue"]
    -
    order_items["quantity"] *
    order_items["unit_selling_price"]
).abs().max()

9.094947017729282e-13

In [93]:
# It should also return 0
(
    order_items["product_cost"]
    -
    order_items["quantity"] *
    order_items["unit_cost"]
).abs().max()

2.2737367544323206e-13

# Shipment Status

In [94]:
#It should also return zero
calculated_status = np.where(
    shipments["delivery_date"]
    <= shipments["promised_date"],
    "On Time",
    "Late"
)

(shipments["delivery_status"] != calculated_status).sum()

np.int64(0)

# Channel Store

In [95]:
# It should also return 0
(
    (orders["sales_channel"] == "Physical Store")
    & orders["store_id"].isna()
).sum()

np.int64(0)

In [97]:
#It should also return 0
(
    (orders["sales_channel"] != "Physical Store")
    & orders["store_id"].notna()
).sum()

np.int64(0)